# AI623 Alignment Repo Runner

Colab-first notebook for SFT, reward modeling, DPO, PPO, GRPO, RLVR, evaluation, plotting, and packaging.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

REPO_MODE = "clone"  # "current" or "clone"
GITHUB_REPO_URL = "https://github.com/Raahim58/Deep-VIsion-Language-Models-"
GITHUB_BRANCH = "main"
USE_DRIVE = False
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

RUN_MODEL_SANITY = True
RUN_ABLATION = True
ABLATION_NAME = "k_sweep"   # one of: kl_sweep, clip_sweep, k_sweep, dpo_beta_sweep
ABLATION_METHOD = "grpo"            # needed only for kl_sweep / clip_sweep

RUN_SFT = True
RUN_RM = True
RUN_DPO = True
RUN_PPO = True
RUN_GRPO = True
RUN_RLVR = True
RUN_EVAL = True

CONFIG_PATHS = ["configs/default.yaml"]
OVERRIDES = {
    "data": {
        "hh_train_samples": 1024,
        "hh_eval_samples": 64,
        "hh_prompt_pool_samples": 256,
        "gsm_train_samples": 512,
        "gsm_eval_samples": 128,
    },
    "ppo": {"steps": 20},
    "grpo": {"steps": 20},
    "rlvr": {"steps": 30},
}

current = Path.cwd().resolve()
candidates = [current, current.parent, current / "code", current.parent / "code"]
repo_root = None

if REPO_MODE == "clone":
    workspace = Path("/content/ai623_alignment_repo")
    if not workspace.exists():
        subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(workspace)], check=True)
    repo_root = workspace / "code" if (workspace / "code").exists() else workspace
else:
    for candidate in candidates:
        if (candidate / "configs" / "default.yaml").exists():
            repo_root = candidate
            break

if repo_root is None:
    raise RuntimeError("Could not locate repo root. Set REPO_MODE='clone' or open the notebook from inside the repo.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Python: {platform.python_version()}")

## Runtime Sanity Checks

This cell checks GPU visibility, VRAM, Python, and torch versions.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Torch version:", torch.__version__)
if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / 1024**3, 2))
else:
    print("Running on CPU. Training will be slow; generation and syntax checks still work.")

## Clone / Mount / Repo Setup

If you want Drive storage, set `USE_DRIVE = True` and run the cell below.

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')
else:
    print('Skipping Drive mount.')

## Dependency Install

In [ ]:
%cd "Programming Assignments/PA2_1/code"
%pip install -q -r requirements.txt

## Hugging Face Login / Environment Variables

Set `HF_TOKEN` in Colab secrets or in the environment before loading the Llama reward/value backbones.

In [ ]:
import os
from huggingface_hub import login

# Try Colab secrets first, then env var, then prompt
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets.')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('HF_TOKEN loaded from environment.')

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in to Hugging Face.')
else:
    print('WARNING: HF_TOKEN not found.')
    print('Reward model and value model require Llama-3.2 access.')
    print('Set HF_TOKEN in Colab Secrets or run: login(token="hf_...")')

## Autoreload Setup

In [ ]:
# %load_ext autoreload
# %autoreload 2

## Imports

In [ ]:
import gc
import json
from pprint import pprint

import pandas as pd
import torch
import yaml
from IPython.display import Image, display

from data.hh_rlhf import load_hh_dataset, make_dpo_dataset, make_prompt_dataset, make_sft_dataset
from data.gsm8k import load_gsm8k_dataset
from model.loading import (
    load_policy_model,
    load_policy_tokenizer,
    load_reference_model,
    load_reward_model,
    load_reward_tokenizer,
    load_value_model,
)
from train_sft import train_sft
from train_rm import train_reward_model
from train_rl import run_dpo, run_ppo, run_grpo, run_rlvr
from eval import evaluate_candidate_vs_reference, evaluate_gsm8k_pass_at_1, evaluate_method_comparison
from utils.config import deep_merge_dicts, load_config
from utils.io import ensure_dir, save_json
from utils.memory import format_parameter_count, get_gpu_report
from utils.plotting import plot_metric_curves

def cleanup_notebook_memory(*var_names):
    for name in var_names:
        if name in globals():
            del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    return get_gpu_report()


def write_runtime_config(config, path):
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as handle:
        yaml.safe_dump(config, handle, sort_keys=False)
    return path


## Config Loading

In [ ]:
config = load_config(CONFIG_PATHS)
config = deep_merge_dicts(config, OVERRIDES)
config["output_dir"] = str(repo_root / "runs")
pprint(config)

## Data Sanity Checks

Print three parsed HH-RLHF examples and confirm prompt/response splitting.

In [ ]:
hh_preview = load_hh_dataset(config, config["data"]["hh_train_split"], max_samples=3)
for idx, row in enumerate(hh_preview):
    print(f"Example {idx}")
    print("PROMPT:")
    print(row["prompt"][-300:])
    print("CHOSEN:")
    print(row["chosen"][:300])
    print("REJECTED:")
    print(row["rejected"][:300])
    print("-" * 80)

## Model Loading Sanity Checks

Loads policy, reference, reward model, and value model once; prints parameter counts and memory report.

In [ ]:
if RUN_MODEL_SANITY:
    policy_tok = load_policy_tokenizer(config["models"]["policy_name"])
    reward_tok = load_reward_tokenizer(config["models"]["reward_name"])
    policy_sanity = load_policy_model(config, trainable=True)
    reference_sanity = load_reference_model(config, checkpoint=config["models"].get("sft_checkpoint"))
    reward_sanity = load_reward_model(config, checkpoint=config["models"].get("rm_checkpoint"), trainable=False)
    value_sanity = load_value_model(config)

    print("Policy tokenizer padding:", policy_tok.padding_side, policy_tok.pad_token_id, policy_tok.eos_token_id)
    print("Reward tokenizer padding:", reward_tok.padding_side, reward_tok.pad_token_id, reward_tok.eos_token_id)
    print("Policy params:", format_parameter_count(policy_sanity))
    print("Reference params:", format_parameter_count(reference_sanity))
    print("Reward params:", format_parameter_count(reward_sanity))
    print("Value params:", format_parameter_count(value_sanity))
    print("GPU report:")
    pprint(get_gpu_report())

    cleanup_notebook_memory("policy_sanity", "reference_sanity", "reward_sanity", "value_sanity")
    pprint(get_gpu_report())
else:
    print("Skipping heavy model sanity load. Set RUN_MODEL_SANITY=True to enable it.")


## Optional SFT

In [ ]:
artifacts = globals().get("artifacts", {})
if RUN_SFT:
    sft_result = train_sft(config)
    artifacts["sft"] = sft_result
    config["models"]["sft_checkpoint"] = sft_result["policy_checkpoint"]
    print(sft_result)
else:
    print("Skipping SFT")

pprint(cleanup_notebook_memory("policy_sanity", "reference_sanity", "reward_sanity", "value_sanity"))


## Optional RM Training

In [ ]:
artifacts = globals().get("artifacts", {})
if RUN_RM:
    rm_result = train_reward_model(config)
    artifacts["rm"] = rm_result
    config["models"]["rm_checkpoint"] = rm_result["rm_checkpoint"]
    print(rm_result)
else:
    print("Skipping RM")

pprint(cleanup_notebook_memory("policy_sanity", "reference_sanity", "reward_sanity", "value_sanity"))


## Optional DPO Training

In [ ]:
artifacts = globals().get("artifacts", {})
if RUN_DPO:
    dpo_result = run_dpo(config)
    artifacts["dpo"] = dpo_result
    print(dpo_result)
else:
    print("Skipping DPO")

pprint(cleanup_notebook_memory("policy_sanity", "reference_sanity", "reward_sanity", "value_sanity"))


## Optional PPO Training

In [ ]:
artifacts = globals().get("artifacts", {})
if RUN_PPO:
    ppo_result = run_ppo(config)
    artifacts["ppo"] = ppo_result
    print(ppo_result)
else:
    print("Skipping PPO")

pprint(cleanup_notebook_memory("policy_sanity", "reference_sanity", "reward_sanity", "value_sanity"))


## Optional GRPO Training

In [ ]:
artifacts = globals().get("artifacts", {})
if RUN_GRPO:
    grpo_result = run_grpo(config)
    artifacts["grpo"] = grpo_result
    print(grpo_result)
else:
    print("Skipping GRPO")

pprint(cleanup_notebook_memory("policy_sanity", "reference_sanity", "reward_sanity", "value_sanity"))


## Optional RLVR Training

In [ ]:
artifacts = globals().get("artifacts", {})
if RUN_RLVR:
    rlvr_result = run_rlvr(config)
    artifacts["rlvr"] = rlvr_result
    print(rlvr_result)
else:
    print("Skipping RLVR")

pprint(cleanup_notebook_memory("policy_sanity", "reference_sanity", "reward_sanity", "value_sanity"))


## Evaluation

In [ ]:
artifacts = globals().get("artifacts", {})
eval_outputs = {}
comparison_outputs = {}
reference_checkpoint = config["models"].get("sft_checkpoint")
if RUN_EVAL and reference_checkpoint:
    eval_metric_rows = []
    for name, meta in artifacts.items():
        if name == "rm":
            continue
        ckpt = meta.get("policy_checkpoint")
        if not ckpt:
            continue
        result = evaluate_candidate_vs_reference(config, ckpt, reference_checkpoint)
        if name == "rlvr":
            result.update(evaluate_gsm8k_pass_at_1(config, ckpt))
        out_dir = Path(ckpt) / "eval"
        ensure_dir(out_dir)
        scalar_result = {k: v for k, v in result.items() if k != "sample_rows"}
        save_json(out_dir / "eval_results.json", scalar_result)
        samples_df = pd.DataFrame(result["sample_rows"])
        samples_df.to_csv(out_dir / "sample_generations.csv", index=False)
        eval_outputs[name] = {"result": result, "out_dir": str(out_dir)}
        eval_metric_rows.append({"method": name, **scalar_result})
    if eval_metric_rows:
        eval_metrics_df = pd.DataFrame(eval_metric_rows)
        print("Evaluation metrics")
        display(eval_metrics_df)
    comparison_candidates = {name: meta["policy_checkpoint"] for name, meta in artifacts.items() if name not in {"rm", "sft"} and meta.get("policy_checkpoint")}
    if comparison_candidates:
        comparison_result = evaluate_method_comparison(config, comparison_candidates, reference_checkpoint, include_gsm8k=("rlvr" in comparison_candidates))
        comparison_dir = repo_root / "runs" / "comparison"
        ensure_dir(comparison_dir)
        comparison_metrics_df = pd.DataFrame(comparison_result["metrics"])
        comparison_resources_df = pd.DataFrame(comparison_result["resource_rows"])
        comparison_samples_df = pd.DataFrame(comparison_result["sample_rows"])
        save_json(comparison_dir / "comparison_metrics.json", {"metrics": comparison_result["metrics"], "resource_rows": comparison_result["resource_rows"]})
        comparison_metrics_df.to_csv(comparison_dir / "comparison_metrics.csv", index=False)
        comparison_resources_df.to_csv(comparison_dir / "resource_table.csv", index=False)
        comparison_samples_df.to_csv(comparison_dir / "comparison_samples.csv", index=False)
        comparison_outputs = {
            "dir": str(comparison_dir),
            "metrics": comparison_result["metrics"],
            "resource_rows": comparison_result["resource_rows"],
            "sample_rows": comparison_result["sample_rows"],
        }
        print("Comparison metrics")
        display(comparison_metrics_df)
        print("Comparison resources")
        display(comparison_resources_df)
        print("Comparison samples")
        display(comparison_samples_df)
    pprint(eval_outputs)
    pprint(comparison_outputs)
else:
    print("Skipping evaluation because RUN_EVAL is False or no SFT reference checkpoint is available.")

pprint(cleanup_notebook_memory())


## Plots / Analysis

In [ ]:
artifacts = globals().get("artifacts", {})
plot_paths = []
for name, meta in artifacts.items():
    run_dir = Path(meta["run_dir"])
    metrics_path = run_dir / "metrics.jsonl"
    if metrics_path.exists():
        plot_path = run_dir / "metrics.png"
        plot_metric_curves(metrics_path, plot_path)
        plot_paths.append(str(plot_path))
        print(f"Plots for {name}: {plot_path}")
        display(Image(filename=str(plot_path)))
plot_paths


## Sample Generations Table

In [ ]:
eval_outputs = globals().get("eval_outputs", {})
comparison_outputs = globals().get("comparison_outputs", {})
for name, payload in eval_outputs.items():
    print(f"Samples for {name}")
    display(pd.DataFrame(payload["result"]["sample_rows"]))
if comparison_outputs.get("sample_rows"):
    print("Comparison sample table")
    display(pd.DataFrame(comparison_outputs["sample_rows"]))


## Focused Ablation

Run exactly one ablation from the current runtime config and SFT checkpoint.


In [ ]:
ablation_outputs = {}
if RUN_ABLATION:
    if not config["models"].get("sft_checkpoint"):
        raise ValueError("Run or set an SFT checkpoint before running ablations.")
    runtime_config_path = write_runtime_config(config, repo_root / "runs" / "notebook_runtime_config.yaml")
    cmd = ["python", "run_ablations.py", "--config", str(runtime_config_path), "--ablation", ABLATION_NAME]
    if ABLATION_NAME in {"kl_sweep", "clip_sweep"}:
        cmd += ["--method", ABLATION_METHOD]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    ablation_dirs = sorted((repo_root / "runs").glob(f"ablation_{ABLATION_NAME}_*"), key=lambda p: p.stat().st_mtime)
    if not ablation_dirs:
        raise FileNotFoundError(f"No ablation output found for {ABLATION_NAME}")
    latest_ablation_dir = ablation_dirs[-1]
    ablation_csv = latest_ablation_dir / "ablation_results.csv"
    ablation_df = pd.read_csv(ablation_csv)
    print("Ablation summary")
    display(ablation_df)

    variant_outputs = []
    for row in ablation_df.to_dict(orient="records"):
        run_dir = Path(row["run_dir"])
        print(f"Variant run: {run_dir.name}")
        display(pd.DataFrame([row]))

        plot_path = run_dir / "metrics.png"
        if plot_path.exists():
            print("Training plot")
            display(Image(filename=str(plot_path)))

        eval_json = run_dir / "eval" / "eval_results.json"
        if eval_json.exists():
            eval_payload = json.loads(eval_json.read_text())
            print("Eval metrics")
            display(pd.DataFrame([eval_payload]))
        else:
            eval_payload = {}

        sample_csv = run_dir / "eval" / "sample_generations.csv"
        if sample_csv.exists():
            sample_df = pd.read_csv(sample_csv)
            print("Sample generations")
            display(sample_df)
        else:
            sample_df = pd.DataFrame()

        variant_outputs.append({
            "run_dir": str(run_dir),
            "plot": str(plot_path) if plot_path.exists() else None,
            "eval": eval_payload,
            "samples_csv": str(sample_csv) if sample_csv.exists() else None,
        })

    ablation_outputs = {
        "run_dir": str(latest_ablation_dir),
        "csv": str(ablation_csv),
        "records": ablation_df.to_dict(orient="records"),
        "variants": variant_outputs,
    }
else:
    print("Skipping ablation.")

pprint(cleanup_notebook_memory())


## Save Outputs / Checkpoints / Metrics

In [ ]:
artifacts = globals().get("artifacts", {})
eval_outputs = globals().get("eval_outputs", {})
comparison_outputs = globals().get("comparison_outputs", {})
ablation_outputs = globals().get("ablation_outputs", {})
plot_paths = globals().get("plot_paths", [])
summary = {
    "repo_root": str(repo_root),
    "artifacts": artifacts,
    "eval_outputs": eval_outputs,
    "comparison_outputs": comparison_outputs,
    "ablation_outputs": ablation_outputs,
    "plots": plot_paths,
}
summary_path = repo_root / "runs" / "notebook_summary.json"
ensure_dir(summary_path.parent)
save_json(summary_path, summary)
print(summary_path)


## Zip Scripts Helper

In [ ]:
!python tools/make_scripts_zip.py
print((repo_root / 'scripts.zip').exists(), repo_root / 'scripts.zip')

## Final Summary

In [ ]:
artifacts = globals().get("artifacts", {})
eval_outputs = globals().get("eval_outputs", {})
comparison_outputs = globals().get("comparison_outputs", {})
ablation_outputs = globals().get("ablation_outputs", {})
print('Completed notebook run.')
print('Artifacts:')
pprint(artifacts)
print('Evaluation outputs:')
pprint(eval_outputs)
print('Comparison outputs:')
pprint(comparison_outputs)
print('Ablation outputs:')
pprint(ablation_outputs)
print('scripts.zip exists:', (repo_root / 'scripts.zip').exists())
